# Notebook 1 — La imagen como matriz: de “foto” a números

Basado en la idea de la presentación de que **la computadora no “ve objetos”, ve una rejilla de números (píxeles)** y que al digitalizar pasamos de una función continua a una matriz discreta. Aquí lo aplicaremos a un ejemplo cotidiano: **medir qué tan iluminado está un cuarto** usando una foto.

**Objetivos didácticos**
- Entender qué es un píxel y por qué una imagen se representa como una matriz.
- Visualizar cómo cambian los valores (0–255) cuando hay más o menos luz.
- Conectar la teoría con una tarea real: *“¿Está bien iluminado mi escritorio?”*.


In [ ]:
# 1) Importamos librerías básicas.
# numpy: para trabajar con matrices (la “imagen” numérica)
# matplotlib: para dibujar imágenes y gráficas
# (1) Importamos librerías/módulos que vamos a usar.
import numpy as np
# (2) Importamos librerías/módulos que vamos a usar.
import matplotlib.pyplot as plt

# (Opcional) Si quieres leer una imagen real, puedes usar PIL.
# (3) Importamos librerías/módulos que vamos a usar.
from PIL import Image

## A) Crear una “imagen” sintética (sin cámara)
Para entenderlo *sin distracciones*, generamos una imagen en escala de grises como matriz.

- 0 significa **negro** (sin luz).
- 255 significa **blanco** (mucha luz).

Construiremos una escena sencilla: un “escritorio” con una lámpara que ilumina más el centro.


In [ ]:
# 2) Definimos el tamaño de la imagen.
# (1) Definimos/asignamos la variable `alto`.
alto = 200   # filas
# (2) Definimos/asignamos la variable `ancho`.
ancho = 300  # columnas

# 3) Creamos una cuadrícula de coordenadas (x, y) para simular un gradiente de luz.
# (3) Definimos/asignamos la variable `y`.
y = np.linspace(-1, 1, alto)[:, None]    # columna (alto x 1)
# (4) Definimos/asignamos la variable `x`.
x = np.linspace(-1, 1, ancho)[None, :]   # fila   (1 x ancho)

# 4) Simulamos una “lámpara” como una mancha gaussiana (brillo mayor cerca del centro).
# (5) Definimos/asignamos la variable `sigma`.
sigma = 0.45
# (6) Definimos/asignamos la variable `brillo`.
brillo = np.exp(-(x**2 + y**2) / (2*sigma**2))

# 5) Convertimos ese brillo (0..1) a niveles de gris (0..255).
# (7) Definimos/asignamos la variable `imagen_gris`.
imagen_gris = (255 * brillo).astype(np.uint8)

# 6) Mostramos la imagen.
# (8) Creamos una nueva figura para graficar.
plt.figure(figsize=(7, 4))
# (9) Mostramos una matriz como imagen.
plt.imshow(imagen_gris, cmap='gray', vmin=0, vmax=255)
# (10) Agregamos un título a la gráfica.
plt.title('Imagen sintética en grises (matriz de 0 a 255)')
# (11) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (12) Renderizamos las gráficas en pantalla.
plt.show()

## B) Ver la imagen como matriz (lo que “ve” la computadora)
Tomemos un pedacito (por ejemplo 5×10 píxeles) y veamos los números.


In [ ]:
# 7) Extraemos un recorte pequeño para imprimirlo.
# (1) Definimos/asignamos la variable `recorte`.
recorte = imagen_gris[90:95, 140:150]
# (2) Ejecutamos esta instrucción como parte del procedimiento.
recorte

### Interpretación intuitiva
- Si el número es grande (cerca de 255), ese píxel es **muy brillante**.
- Si el número es pequeño (cerca de 0), ese píxel es **oscuro**.

Esta es exactamente la idea del PDF: **percepción vs matriz**.


## C) Medición cotidiana: ¿qué tan iluminado está el “escritorio”?
En un caso real, podrías tomar una foto de tu escritorio. Un proxy simple es calcular:
- **Promedio de intensidad** en una región (ROI, región de interés).

Esto se usa en industria para control de iluminación y calidad de captura.


In [ ]:
# 8) Definimos una región de interés (ROI): una ventana al centro.
# (1) Definimos/asignamos la variable `roi`.
roi = imagen_gris[70:130, 110:190]

# 9) Calculamos estadísticas básicas.
# (2) Definimos/asignamos la variable `promedio`.
promedio = float(np.mean(roi))
# (3) Definimos/asignamos la variable `minimo`.
minimo   = int(np.min(roi))
# (4) Definimos/asignamos la variable `maximo`.
maximo   = int(np.max(roi))

# (5) Ejecutamos esta instrucción como parte del procedimiento.
promedio, minimo, maximo

In [ ]:
# 10) Visualizamos la ROI para entender qué estamos midiendo.
# (1) Creamos una nueva figura para graficar.
plt.figure(figsize=(5, 4))
# (2) Mostramos una matriz como imagen.
plt.imshow(roi, cmap='gray', vmin=0, vmax=255)
# (3) Agregamos un título a la gráfica.
plt.title(f'ROI (promedio={promedio:.1f}, min={minimo}, max={maximo})')
# (4) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (5) Renderizamos las gráficas en pantalla.
plt.show()

## D) Cuantización: ¿qué pasa si guardo la imagen con menos bits?
El PDF menciona **profundidad de bits** y que 8 bits (0–255) es estándar.

Aquí simulamos bajar la cantidad de niveles (por ejemplo 4 bits → 16 niveles) y veremos “bandas”.


In [ ]:
# 11) Función para cuantizar a L niveles.
# (1) Definimos la función `cuantizar`.
def cuantizar(im, L):
    # im: imagen uint8 0..255
    # L: niveles discretos (p.ej., 16)
    # (2) Definimos/asignamos la variable `im_f`.
    im_f = im.astype(np.float32)
    # (3) Definimos/asignamos la variable `paso`.
    paso = 255 / (L - 1)                # tamaño de escalón
    # (4) Definimos/asignamos la variable `im_q`.
    im_q = np.round(im_f / paso) * paso # redondeamos al nivel más cercano
    # (5) Convertimos el tipo de dato (por ejemplo a float o uint8).
    return im_q.astype(np.uint8)

# (6) Definimos/asignamos la variable `img_8bit`.
img_8bit = imagen_gris
# (7) Definimos/asignamos la variable `img_4bit`.
img_4bit = cuantizar(imagen_gris, L=16)
# (8) Definimos/asignamos la variable `img_2bit`.
img_2bit = cuantizar(imagen_gris, L=4)

# (9) Creamos una nueva figura para graficar.
plt.figure(figsize=(12, 4))
# (10) Seleccionamos una zona (subplot) dentro de la figura.
plt.subplot(1, 3, 1)
# (11) Mostramos una matriz como imagen.
plt.imshow(img_8bit, cmap='gray', vmin=0, vmax=255)
# (12) Agregamos un título a la gráfica.
plt.title('8-bit (256 niveles)')
# (13) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')

# (14) Seleccionamos una zona (subplot) dentro de la figura.
plt.subplot(1, 3, 2)
# (15) Mostramos una matriz como imagen.
plt.imshow(img_4bit, cmap='gray', vmin=0, vmax=255)
# (16) Agregamos un título a la gráfica.
plt.title('4-bit (16 niveles)')
# (17) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')

# (18) Seleccionamos una zona (subplot) dentro de la figura.
plt.subplot(1, 3, 3)
# (19) Mostramos una matriz como imagen.
plt.imshow(img_2bit, cmap='gray', vmin=0, vmax=255)
# (20) Agregamos un título a la gráfica.
plt.title('2-bit (4 niveles)')
# (21) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (22) Renderizamos las gráficas en pantalla.
plt.show()

### Conclusión práctica
En fotos de celular generalmente trabajas con 8 bits por canal (RGB). Para medicina o satélite, se usan 12–16 bits porque necesitas distinguir cambios muy finos de brillo.

Esto conecta con **radiométrica** y **profundidad de bits** de la presentación.
